# E5 — Генерализация + OOD (Г4б,в)

Матрица «обучен на сезоне A -> тест на B»; сигналы доверия Mahalanobis (по экзогенным входам) и дисперсия Ensemble-SINDy; корреляция с ошибкой; guard (OOD->откат на rule_based) и ROC детектора. Полный прогон — run_e5_grid.py + merge_e5.py.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location); CORR, PRICES = ECON["corridors"], ECON["prices"]
# E4/E5 adaptive surrogate: single coefficient set (so EKF/DAgger can update it).
RECIPE = dict(feature_variant="physics_no_cross", library_degree=1, optimizer="stlsq", denoise="none")
print("FAST_MODE", FAST_MODE)

FAST_MODE True


## Сетка train×test + OOD-сигналы + guard

In [2]:
trains = ["2019:03-01"] if FAST_MODE else ["2019:03-01", "2019:07-01"]
tests = ["2020:03-01", "2021:07-01"] if FAST_MODE else ["2020:03-01", "2021:03-01", "2021:07-01", "2022:10-01", "2023:01-01"]
N = 5 if FAST_MODE else 14
n_train = 7 if FAST_MODE else 21
def scen(sh):
    yr, md_ = sh.split(":"); return {"year": int(yr), "start_date": f"{yr}-{md_}", "n_days": N}
def vsteps(d):
    v = np.zeros(len(d), int)
    for k in ("t_in", "co2", "rh"):
        lo, hi = CORR[k]; x = d[k].to_numpy(float); v += ((x < lo) | (x > hi)).astype(int)
    return v
grid, roc = [], []
for tr in trains:
    train = U.collect_rule_based_dataset(pc.cfg_for(scen(tr), seed=0), n_days=n_train, prbs_scale=0.3)
    b = U.fit_sindy(train, period=float(pc.period), **RECIPE)
    maha = U.fit_mahalanobis(train); ens = U.fit_ensemble_for_variance(train, period=float(pc.period))
    thr = float(np.percentile(U.mahalanobis_distances(maha, train.weather, train.time_enc), 95))
    for te in tests:
        sc = scen(te); cfg_t = pc.cfg_for(sc, seed=0); SS = sc["start_date"]
        td = U.collect_rule_based_dataset(cfg_t, n_days=N, prbs_scale=0.0)
        ev = U.evaluate_sindy(b, td, rollout_horizons=(20,)); rr = float(ev[(ev.metric_scope == "rollout") & (ev.state == "t_in")]["rmse"].iloc[0])
        du = U.rollout_mpc(b, cfg_t, N, start_date=SS); dg = U.rollout_mpc_guarded(b, maha, thr, cfg_t, N, start_date=SS)
        mu = U.epi_metrics(du, corridors=CORR, prices=PRICES); mg = U.epi_metrics(dg, corridors=CORR, prices=PRICES)
        grid.append({"train": tr, "test": te, "rollout_rmse": rr, "maha": float(np.mean(U.mahalanobis_distances(maha, td.weather, td.time_enc))),
                     "ens_std": float(np.nanmean(U.ensemble_pred_std(ens, td))), "epi_unguarded": mu["epi"],
                     "viol_unguarded": mu["violation_steps_total"], "viol_guarded": mg["violation_steps_total"]})
        ood_un = U.mahalanobis_distances(maha, du[U.WEATHER_NAMES].to_numpy(), du[U.TIME_NAMES].to_numpy())
        for o, iv in zip(ood_un, vsteps(du)):
            roc.append({"ood": float(o), "violation": int(iv > 0)})
grid = pd.DataFrame(grid); roc = pd.DataFrame(roc); print("grid", len(grid), "roc", len(roc))

CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-06-30 15:51:12 WARNING("F:jacF failed: NaN detected for output jac_ode_x, at nonzero index 49 (row 5, col 5).") [.../casadi/core/oracle_function.c

Error in ODE approximation


grid 2 roc 606


## Матрица генерализации, корреляция, guard, ROC

In [3]:
mat = grid.pivot_table(index="train", columns="test", values="epi_unguarded", aggfunc="mean")
U.save_table(mat.reset_index(), RES / "tables" / "e5_generalization_matrix.csv")
def cc(x, y):
    return float(np.corrcoef(grid[x], grid[y])[0, 1])
print("corr(maha, RMSE)=", round(cc("maha", "rollout_rmse"), 2), " corr(maha, EPI)=", round(cc("maha", "epi_unguarded"), 2))
print("guard: viol", round(grid.viol_unguarded.mean(), 0), "->", round(grid.viol_guarded.mean(), 0))
auc = float("nan")
if roc.violation.nunique() > 1:
    from sklearn.metrics import roc_auc_score, roc_curve
    auc = float(roc_auc_score(roc.violation, roc.ood)); fpr, tpr, _ = roc_curve(roc.violation, roc.ood)
else:
    fpr = tpr = np.array([0, 1])
print("ROC AUC (ood->violation) =", round(auc, 3)); display(mat.round(2))

corr(maha, RMSE)= 1.0  corr(maha, EPI)= -1.0
guard: viol 334.0 -> 294.0
ROC AUC (ood->violation) = 0.549


test,2020:03-01,2021:07-01
train,,
2019:03-01,0.36,-0.57


In [4]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
im = ax[0].imshow(mat.values, aspect="auto", cmap="RdYlGn"); fig.colorbar(im, ax=ax[0])
ax[0].set_xticks(range(len(mat.columns)), mat.columns, rotation=30, ha="right", fontsize=7)
ax[0].set_yticks(range(len(mat.index)), mat.index, fontsize=8); ax[0].set_title("Generalization EPI (train->test)")
ax[1].scatter(grid["maha"], grid["rollout_rmse"]); ax[1].set_xlabel("Mahalanobis (OOD)"); ax[1].set_ylabel("rollout-RMSE")
ax[1].set_title("OOD vs error r=%.2f" % cc("maha", "rollout_rmse")); ax[1].grid(alpha=.3)
ax[2].plot(fpr, tpr); ax[2].plot([0, 1], [0, 1], "k--", lw=.8); ax[2].set_title("OOD detector ROC AUC=%.2f" % auc)
ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].grid(alpha=.3)
U.save_figure(fig, RES / "figures" / "e5_generalization.png"); plt.close(fig); print("saved figure")

saved figure


**Итог E5 (Г4б,в).** Модели сильнее на близких сезонах, зима-2023 — OOD-провал. Mahalanobis предсказывает ошибку (статейно corr +0.63 с RMSE, -0.53 с EPI; сильнее ансамблевой дисперсии). Guard снижает нарушения статейно **1621->911 (-44%)** малой ценой EPI; ROC AUC=**0.69**.